# Exercise 01 — API Basics

This notebook covers the foundational skills for working with the Claude API:

1. Setting up your environment
2. Making your first API request
3. Multi-turn conversations (keeping chat history)
4. System prompts (giving Claude a persona or role)
5. Temperature (controlling creativity)

**Prerequisites:** Create a `.env` file one level up from this notebook with:
```
ANTHROPIC_API_KEY="your_key_here"
```

## Part 1 — Setup

Install the required packages (only needs to run once).

In [ ]:
# Run this once to install dependencies
# !pip install anthropic python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
import anthropic

# Load API key from .env file
load_dotenv(dotenv_path="../.env")

client = anthropic.Anthropic()  # automatically reads ANTHROPIC_API_KEY from environment
MODEL = "claude-haiku-4-5"      # using Haiku for exercises (fast + cheap)

print("Client created successfully!")

## Part 2 — Your First API Request

Every API request needs three required arguments:
- `model` — which Claude model to use
- `max_tokens` — the maximum number of tokens Claude can generate (an upper limit, not a target)
- `messages` — the conversation as a list of `{role, content}` dicts

The response object contains more than just text. Let's explore it.

In [ ]:
# Make a basic API request
response = client.messages.create(
    model=MODEL,
    max_tokens=512,
    messages=[
        {"role": "user", "content": "What is the capital of France? Answer in one sentence."}
    ]
)

# Inspect the full response
print("--- Full Response ---")
print(response)
print()
print("--- Just the text ---")
print(response.content[0].text)
print()
print("--- Usage info ---")
print(f"Input tokens:  {response.usage.input_tokens}")
print(f"Output tokens: {response.usage.output_tokens}")
print(f"Stop reason:   {response.stop_reason}")

### Exercise 2a

Modify the request above to ask Claude: **"Name three programming languages and explain each in one sentence."**

Try two different `max_tokens` values (e.g. `100` and `500`). What changes?

In [ ]:
# TODO: Write your request here
response = client.messages.create(
    model=MODEL,
    max_tokens=100,   # try changing this to 500
    messages=[
        {"role": "user", "content": "Name three programming languages and explain each in one sentence."}
    ]
)

print(response.content[0].text)
print(f"\nStop reason: {response.stop_reason}")

## Part 3 — Multi-Turn Conversations

**Key limitation:** Claude has NO memory between requests. Each `messages.create()` call is independent.

**Solution:** You manage a `messages` list yourself and send the **entire history** with every new request.

Let's build simple helper functions and a chat loop.

In [ ]:
def add_user_message(messages: list, text: str) -> list:
    """Append a user message to the conversation history."""
    messages.append({"role": "user", "content": text})
    return messages


def add_assistant_message(messages: list, text: str) -> list:
    """Append an assistant message to the conversation history."""
    messages.append({"role": "assistant", "content": text})
    return messages


def chat(messages: list, system: str = None) -> str:
    """Send the full message history to Claude and return the reply text."""
    params = {
        "model": MODEL,
        "max_tokens": 1024,
        "messages": messages,
    }
    if system:
        params["system"] = system
    response = client.messages.create(**params)
    return response.content[0].text

In [ ]:
# Demo: multi-turn conversation that remembers context
messages = []

# Turn 1
add_user_message(messages, "My favourite colour is blue and my name is Alex.")
reply = chat(messages)
add_assistant_message(messages, reply)
print(f"Claude: {reply}\n")

# Turn 2 — Claude should remember both facts
add_user_message(messages, "What is my name and what colour did I mention?")
reply = chat(messages)
add_assistant_message(messages, reply)
print(f"Claude: {reply}\n")

print(f"Messages in history: {len(messages)}")

---
## Chat Bot Exercise

> **Task:** Make a chat bot using the three helper functions above.
>
> 1. Prompt the user to enter some input using the built-in `input()` function
> 2. Add it to a list of messages
> 3. Call the API
> 4. Add generated text to the list of messages
> 5. Print the generated text
> 6. Repeat from #1

Type `quit` or `exit` to stop the conversation.

In [ ]:
def run_chatbot(system_prompt: str = None) -> None:
    """Interactive chat bot loop using the three helper functions."""
    messages = []  # conversation history starts empty
    print("Chat bot ready! Type 'quit' or 'exit' to stop.\n")

    while True:
        # Step 1 — prompt the user for input
        user_input = input("You: ").strip()

        if user_input.lower() in ("quit", "exit", ""):
            print("Goodbye!")
            break

        # Step 2 — add user input to the message list
        add_user_message(messages, user_input)

        # Step 3 — call the API
        reply = chat(messages, system=system_prompt)

        # Step 4 — add generated text to the message list
        add_assistant_message(messages, reply)

        # Step 5 — print the generated text
        print(f"\nAssistant: {reply}\n")

        # Step 6 — loop repeats automatically


# Run the chat bot
run_chatbot()

### Exercise 3a — Without History

Prove that Claude has no memory by sending the follow-up question as a **fresh request** (empty messages list). Compare the result.

In [ ]:
# Fresh request — no history
fresh_messages = []
add_user_message(fresh_messages, "What is my name and what colour did I mention?")
reply = chat(fresh_messages)
print(f"Claude (no history): {reply}")

## Part 4 — System Prompts

System prompts tell Claude **how to behave** — they define its persona, tone, and rules. They are passed separately from the user messages using the `system` parameter.

The same user question can get very different answers depending on the system prompt.

In [ ]:
MATH_TUTOR_SYSTEM = """You are a specialist math tutor.

Responses SHOULD:
- Initially, only give the student hints -- do not reveal the answer straight away
- Patiently walk the student through the solution step by step when they need more help
- Show solutions for similar (but different) problems as examples

Responses should NOT:
- Immediately give a direct answer to the student's question
- Tell the student to use a calculator"""

user_question = "How do I solve 5x + 2 = 3 for x?"

# Without system prompt -- gives the answer directly
print("=== Without system prompt ===")
plain_messages = [{"role": "user", "content": user_question}]
print(chat(plain_messages))

print()

# With Math Tutor Specialist system prompt -- gives hints first
print("=== With Math Tutor Specialist system prompt ===")
tutor_messages = [{"role": "user", "content": user_question}]
print(chat(tutor_messages, system=MATH_TUTOR_SYSTEM))

In [ ]:
# Run an interactive Math Tutor Specialist chat session
# Ask it a maths problem and see how it guides you with hints instead of direct answers
run_chatbot(system_prompt=MATH_TUTOR_SYSTEM)

---
## Exercise — System Prompts

A system prompt controls **how** Claude responds — its persona, tone, rules, and restrictions — without changing what the user asks.

**Key rule:** The system prompt is passed separately using the `system` parameter, not inside the `messages` list.

### Part A — Compare personas on the same question

The cell below sends the same user question to three different "versions" of Claude:
1. No system prompt (default behaviour)
2. A formal corporate assistant
3. A casual, emoji-friendly assistant

Notice how the same question gets completely different treatment.

In [ ]:
QUESTION = "What's the best way to stay productive when working from home?"

FORMAL_SYSTEM = """You are a corporate productivity consultant writing for a Fortune 500 company.
Use formal business language. Structure your response with numbered points.
Keep responses under 100 words."""

CASUAL_SYSTEM = """You are a fun, upbeat life coach who loves using emojis.
Keep it short, punchy, and motivating. Max 3 bullet points."""

personas = [
    ("No system prompt",        None),
    ("Corporate consultant",    FORMAL_SYSTEM),
    ("Casual life coach",       CASUAL_SYSTEM),
]

for name, system in personas:
    print(f"\n{'='*55}")
    print(f"  {name}")
    print('='*55)
    messages = [{"role": "user", "content": QUESTION}]
    print(chat(messages, system=system))

### Part B — Using system prompts to restrict behaviour

System prompts can enforce rules — what Claude should and should NOT do.
This is exactly what we did with the Math Tutor Specialist above.

Try the two prompts below and observe how the restrictions hold up even when the user pushes back.

In [ ]:
SUPPORT_SYSTEM = """You are a customer support agent for Acme Software.

You SHOULD:
- Answer questions about Acme Software products only
- Be polite and empathetic
- Escalate billing issues to billing@acme.com

You should NOT:
- Discuss competitor products
- Answer questions unrelated to Acme Software
- Make up features that don't exist"""

test_messages = [
    "How do I reset my Acme account password?",          # on-topic
    "Can you recommend a good Italian restaurant nearby?", # off-topic
    "Is your product better than CompetitorX?",            # restricted topic
]

for msg in test_messages:
    print(f"\nUser: {msg}")
    messages = [{"role": "user", "content": msg}]
    print(f"Assistant: {chat(messages, system=SUPPORT_SYSTEM)}")

### Part C — Write your own system prompt

Design a system prompt for one of these roles (or invent your own):

- A code reviewer that only comments on security issues
- A recipe assistant that only suggests 5-ingredient meals
- A travel guide for a specific city

Fill in `MY_SYSTEM_PROMPT` and `MY_QUESTION` below, then run the cell.

In [ ]:
MY_SYSTEM_PROMPT = """TODO: Write your system prompt here.

Think about:
- What role should Claude play?
- What should it always do?
- What should it never do?
- What tone/style should it use?"""

MY_QUESTION = "TODO: Write a test question here."

messages = [{"role": "user", "content": MY_QUESTION}]
print(chat(messages, system=MY_SYSTEM_PROMPT))

# Tip: once it works as a single response, try the interactive version:
# run_chatbot(system_prompt=MY_SYSTEM_PROMPT)

## Part 5 — Temperature

Temperature controls how predictable or creative Claude's responses are:

- **0** → always picks the most likely next word → deterministic, consistent outputs
- **1** → picks more surprising words → creative, varied outputs

Run the same creative prompt multiple times at different temperatures to see the difference.

In [ ]:
def chat_with_temperature(prompt: str, temperature: float, runs: int = 3) -> None:
    """Call Claude multiple times with the same prompt and temperature."""
    print(f"\n{'='*50}")
    print(f"Temperature: {temperature}")
    print('='*50)
    for i in range(runs):
        response = client.messages.create(
            model=MODEL,
            max_tokens=100,
            temperature=temperature,
            messages=[{"role": "user", "content": prompt}]
        )
        print(f"Run {i+1}: {response.content[0].text.strip()}")


prompt = "Write one sentence describing the colour blue in a creative way."

chat_with_temperature(prompt, temperature=0.0)   # deterministic
chat_with_temperature(prompt, temperature=0.5)   # balanced
chat_with_temperature(prompt, temperature=1.0)   # creative

### Exercise 5a — Pick the Right Temperature

For each task below, choose an appropriate temperature and explain why:

1. Extract the dates from a news article → temperature = ___  (reason: ___)
2. Write five different taglines for a shampoo brand → temperature = ___  (reason: ___)
3. Answer a maths question → temperature = ___  (reason: ___)

Then test your choices in the cell below.

In [ ]:
# Task 1 — data extraction
extraction_response = client.messages.create(
    model=MODEL,
    max_tokens=200,
    temperature=0.0,  # TODO: set appropriate temperature
    messages=[{"role": "user", "content": 
        "Extract all dates from this text: 'The conference ran from June 3 to June 5 2024. "
        "Registration closed on May 15 2024.'"}]
)
print("Data extraction:")
print(extraction_response.content[0].text)

print()

# Task 2 — creative writing
creative_response = client.messages.create(
    model=MODEL,
    max_tokens=200,
    temperature=1.0,  # TODO: set appropriate temperature
    messages=[{"role": "user", "content": "Write five taglines for a shampoo brand."}]
)
print("Creative taglines:")
print(creative_response.content[0].text)

## Summary

You now know how to:

- ✅ Set up the Anthropic client from an environment variable
- ✅ Make API requests with `model`, `max_tokens`, `messages`
- ✅ Build a multi-turn conversation by maintaining a messages list
- ✅ Use system prompts to give Claude a persona or behavioral rules
- ✅ Control output creativity with the `temperature` parameter

**Next:** [02_streaming_output_control.ipynb](02_streaming_output_control.ipynb) — streaming responses, pre-filling, stop sequences, and structured JSON extraction